# 作业2 — 地域与交易额分析（member2）

目标：基于清洗后的 Google Analytics 数据，统计不同**国家/大洲**的交易额（Revenue），并进行可视化展示（柱状图 + 世界地图），同时导出可复用的汇总表。

约束：**只允许增加和查询**，所有产物写入 `DataIntegration-hw2/member2/` 下的子目录，且默认使用**时间戳文件名**避免覆盖已有文件。

In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import pandas as pd

# 可视化
import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120


## 1. 创建输出目录（member2/tables, figures, logs）

- 只新增文件：所有导出文件名都附带时间戳，默认不覆盖已有文件
- 目录：`DataIntegration-hw2/member2/{tables,figures,logs}`

In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "DataIntegration-hw2").exists() and ((p / "as2_data").exists() or (p / "as_data").exists()):
            return p
    # fallback
    return start


REPO_ROOT = find_repo_root()
MEMBER2_DIR = REPO_ROOT / "DataIntegration-hw2" / "member2"
TABLES_DIR = MEMBER2_DIR / "tables"
FIG_DIR = MEMBER2_DIR / "figures"
LOG_DIR = MEMBER2_DIR / "logs"

for d in [TABLES_DIR, FIG_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
print("REPO_ROOT =", REPO_ROOT)
print("Output base =", MEMBER2_DIR)


def unique_path(dir_path: Path, stem: str, suffix: str) -> Path:
    # Always create a new file path; never overwrite
    p = dir_path / f"{stem}_{run_ts}{suffix}"
    if not p.exists():
        return p

    # extremely unlikely, but keep it safe
    i = 1
    while True:
        alt = dir_path / f"{stem}_{run_ts}_{i}{suffix}"
        if not alt.exists():
            return alt
        i += 1


def write_json(obj: dict, dir_path: Path, stem: str) -> Path:
    path = unique_path(dir_path, stem, ".json")
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path


def write_csv(df: pd.DataFrame, dir_path: Path, stem: str) -> Path:
    path = unique_path(dir_path, stem, ".csv")
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return path


## 2. 从 ./as_data/cleaned 加载清洗数据（自动发现 + Schema 检查）

说明：本 workspace 中实际存在 `as2_data/cleaned/`，同时为了兼容作业描述也会尝试 `as_data/cleaned/`；以实际存在的目录为准。

In [ ]:
def find_cleaned_dir(repo_root: Path) -> Path:
    for rel in [Path("as_data/cleaned"), Path("as2_data/cleaned")]:
        p = repo_root / rel
        if p.exists():
            return p
    raise FileNotFoundError("Cannot find cleaned data dir: tried as_data/cleaned and as2_data/cleaned")


CLEANED_DIR = find_cleaned_dir(REPO_ROOT)
print("CLEANED_DIR =", CLEANED_DIR)

all_files = sorted([p.name for p in CLEANED_DIR.glob("*.csv")])
print("Discovered files:")
for fn in all_files:
    print("-", fn)

# Load key tables needed for region + revenue analysis
required = {
    "ga_session.csv": ["uid", "full_visitor_id", "date"],
    "ga_total.csv": ["uid", "transactions", "total_transaction_revenue", "visits"],
    "ga_geo_network.csv": ["uid", "country", "continent", "sub_continent", "region", "city"],
}

missing = [fn for fn in required if not (CLEANED_DIR / fn).exists()]
if missing:
    raise FileNotFoundError(f"Missing required cleaned files: {missing}")

sessions = pd.read_csv(CLEANED_DIR / "ga_session.csv")
totals = pd.read_csv(CLEANED_DIR / "ga_total.csv")
geo = pd.read_csv(CLEANED_DIR / "ga_geo_network.csv")

print("\nRow counts:")
print("ga_session:", len(sessions))
print("ga_total:", len(totals))
print("ga_geo_network:", len(geo))


def data_dictionary(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = pd.DataFrame({
        "table": name,
        "column": df.columns,
        "dtype": [str(t) for t in df.dtypes],
        "null_rate": df.isna().mean().values,
    })
    return out


dict_df = pd.concat(
    [data_dictionary(sessions, "ga_session"), data_dictionary(totals, "ga_total"), data_dictionary(geo, "ga_geo_network")],
    ignore_index=True,
)

schema_path = write_csv(dict_df, LOG_DIR, "data_dictionary")
print("\nSaved data dictionary ->", schema_path)

used_files_log = {
    "cleaned_dir": str(CLEANED_DIR),
    "files_used": list(required.keys()),
}
log_path = write_json(used_files_log, LOG_DIR, "input_files")
print("Saved input files log ->", log_path)


## 3. 识别交易指标（Revenue）与地域字段（Country/Continent）

形式化定义（以国家为例）：

$$R_c = \sum_{i \in c} revenue_i$$

其中 $i$ 是 session（以 `uid` 为键），$revenue_i$ 来自总表中的交易额字段。

In [ ]:
def pick_first_existing(df: pd.DataFrame, candidates: list[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of the candidates exist in dataframe: {candidates}")


# Candidate names (keep it flexible)
uid_col = pick_first_existing(sessions, ["uid", "session_id", "sessionId"])
visitor_col = pick_first_existing(sessions, ["full_visitor_id", "fullVisitorId", "visitor_id"])
date_col = pick_first_existing(sessions, ["date", "visit_date"])

revenue_col = pick_first_existing(totals, ["total_transaction_revenue", "transactionRevenue", "revenue", "totalRevenue"])
transactions_col = pick_first_existing(totals, ["transactions", "orders", "transaction_count"])

country_col = pick_first_existing(geo, ["country", "geoNetwork_country"])
continent_col = pick_first_existing(geo, ["continent", "geoNetwork_continent"])

print("Chosen columns:")
print("- uid:", uid_col)
print("- visitor:", visitor_col)
print("- date:", date_col)
print("- revenue:", revenue_col)
print("- transactions:", transactions_col)
print("- country:", country_col)
print("- continent:", continent_col)

# Join at session grain
df = (
    sessions[[uid_col, visitor_col, date_col]]
    .merge(totals[[uid_col, revenue_col, transactions_col]], on=uid_col, how="left")
    .merge(geo[[uid_col, country_col, continent_col]], on=uid_col, how="left")
)

# Parse and clean
_df = df.copy()
_df["date"] = pd.to_datetime(_df[date_col].astype(str), format="%Y%m%d", errors="coerce")
_df["revenue_raw"] = pd.to_numeric(_df[revenue_col], errors="coerce")
_df["transactions"] = pd.to_numeric(_df[transactions_col], errors="coerce")
_df["country"] = _df[country_col].astype(str).str.strip()
_df["continent"] = _df[continent_col].astype(str).str.strip()

# Normalize placeholders
for col in ["country", "continent"]:
    _df.loc[_df[col].isin(["nan", "None", "(not set)", "not available in demo dataset"]), col] = pd.NA

# Revenue unit conversion: course handout suggests micros
_df["revenue_usd"] = _df["revenue_raw"] / 1_000_000.0

# Simple validations
if (_df["revenue_usd"].dropna() < 0).any():
    raise ValueError("Found negative revenue after conversion; please re-check revenue column")

print("\nBasic stats:")
print("- sessions:", len(_df))
print("- date range:", _df["date"].min(), "~", _df["date"].max())
print("- unique countries:", _df["country"].nunique(dropna=True))
print("- paid sessions (revenue>0):", int((_df["revenue_usd"].fillna(0) > 0).sum()))
print("- total revenue (USD):", float(_df["revenue_usd"].fillna(0).sum()))

chosen_cols_log = {
    "uid": uid_col,
    "visitor": visitor_col,
    "date": date_col,
    "revenue": revenue_col,
    "transactions": transactions_col,
    "country": country_col,
    "continent": continent_col,
    "revenue_unit": "micros_to_usd_divide_1e6",
}
log_path = write_json(chosen_cols_log, LOG_DIR, "chosen_columns")
print("Saved chosen columns log ->", log_path)


## 4. 国家维度聚合：总交易额 / 订单数 / 用户数（Top-N）

输出：
- Top10/Top20 国家表（按交易额、按交易数）
- 额外计算：平均每笔交易金额（`revenue_usd / transactions`）

In [ ]:
paid = _df.loc[_df["revenue_usd"].fillna(0) > 0].copy()

country_summary = (
    paid.groupby("country", dropna=True)
    .agg(
        revenue_usd=("revenue_usd", "sum"),
        transactions=("transactions", "sum"),
        sessions=(uid_col, "count"),
        visitors=(visitor_col, pd.Series.nunique),
    )
    .reset_index()
)

country_summary["avg_revenue_per_txn_usd"] = country_summary["revenue_usd"] / country_summary["transactions"].replace({0: pd.NA})

country_by_revenue = country_summary.sort_values("revenue_usd", ascending=False)
country_by_txn = country_summary.sort_values("transactions", ascending=False)

print("Top 10 by revenue:")
display(country_by_revenue.head(10))

print("\nTop 10 by transactions:")
display(country_by_txn.head(10))

path_all = write_csv(country_by_revenue, TABLES_DIR, "country_summary_by_revenue")
path_top20_rev = write_csv(country_by_revenue.head(20), TABLES_DIR, "country_top20_by_revenue")
path_top20_txn = write_csv(country_by_txn.head(20), TABLES_DIR, "country_top20_by_transactions")

print("\nSaved tables:")
print("-", path_all)
print("-", path_top20_rev)
print("-", path_top20_txn)


## 5. 国家→大洲映射与大洲维度聚合

这里优先使用数据集中 `ga_geo_network` 已提供的 `continent` 字段；同时导出一个 `country -> continent` 的映射表，便于报告撰写与复现。

In [ ]:
# Build a stable mapping from observed data (most frequent continent per country)
country_continent_map = (
    paid.dropna(subset=["country", "continent"])
    .groupby(["country", "continent"], as_index=False)
    .size()
    .sort_values(["country", "size"], ascending=[True, False])
    .drop_duplicates("country")
    .drop(columns=["size"])
    .sort_values("country")
    .reset_index(drop=True)
)

map_path = write_csv(country_continent_map, TABLES_DIR, "country_to_continent_mapping")
print("Saved mapping ->", map_path)

continent_summary = (
    paid.groupby("continent", dropna=True)
    .agg(
        revenue_usd=("revenue_usd", "sum"),
        transactions=("transactions", "sum"),
        sessions=(uid_col, "count"),
        visitors=(visitor_col, pd.Series.nunique),
    )
    .reset_index()
    .sort_values("revenue_usd", ascending=False)
)

continent_summary["revenue_share"] = continent_summary["revenue_usd"] / continent_summary["revenue_usd"].sum()

display(continent_summary)

continent_path = write_csv(continent_summary, TABLES_DIR, "continent_summary_by_revenue")
print("Saved continent summary ->", continent_path)


## 6. 可视化：国家交易额柱状图 + 帕累托（累计占比）

累计占比定义：

$$s_k = \frac{\sum_{j=1}^k R_{(j)}}{\sum_j R_j}$$

其中 $R_{(j)}$ 表示按交易额从高到低排序后的第 $j$ 个国家交易额。

In [ ]:
top_n = 10
plot_df = country_by_revenue.head(top_n).copy()

# cumulative share (on all countries, not just top_n)
all_sorted = country_by_revenue.copy()
all_sorted["cum_revenue"] = all_sorted["revenue_usd"].cumsum()
all_sorted["cum_share"] = all_sorted["cum_revenue"] / all_sorted["revenue_usd"].sum()

plot_df = plot_df.merge(all_sorted[["country", "cum_share"]], on="country", how="left")

fig, ax1 = plt.subplots(figsize=(12, 6))

sns.barplot(data=plot_df, x="country", y="revenue_usd", ax=ax1, color=sns.color_palette()[0])
ax1.set_title(f"Top {top_n} Countries by Revenue + Cumulative Share")
ax1.set_xlabel("Country")
ax1.set_ylabel("Revenue (USD)")
ax1.tick_params(axis="x", rotation=45, ha="right")

ax2 = ax1.twinx()
ax2.plot(plot_df["country"], plot_df["cum_share"], color=sns.color_palette()[3], marker="o")
ax2.set_ylabel("Cumulative Share")
ax2.set_ylim(0, 1.05)
ax2.grid(False)

for i, v in enumerate(plot_df["revenue_usd"].values):
    ax1.text(i, v, f"{v:.0f}", ha="center", va="bottom", fontsize=9)

for i, s in enumerate(plot_df["cum_share"].values):
    ax2.text(i, s, f"{s:.0%}", ha="center", va="bottom", fontsize=9, color=sns.color_palette()[3])

plt.tight_layout()
fig_path = unique_path(FIG_DIR, f"top{top_n}_countries_revenue_pareto", ".png")
plt.savefig(fig_path, dpi=200)
plt.close(fig)

print("Saved figure ->", fig_path)


## 7. 可视化：大洲交易额柱状图

在柱状图上标注占全球交易额的比例，便于报告解读。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(data=continent_summary, y="continent", x="revenue_usd", ax=ax, color=sns.color_palette()[1])
ax.set_title("Continents by Total Revenue")
ax.set_xlabel("Revenue (USD)")
ax.set_ylabel("Continent")

for i, row in continent_summary.reset_index(drop=True).iterrows():
    ax.text(row["revenue_usd"], i, f"  {row['revenue_share']:.0%}", va="center")

plt.tight_layout()
fig_path = unique_path(FIG_DIR, "continents_revenue", ".png")
plt.savefig(fig_path, dpi=200)
plt.close(fig)

print("Saved figure ->", fig_path)


## 8. 可视化：世界交易额地图（Choropleth）

使用 Plotly Express 生成交互式 HTML（便于报告截图/展示）；并尝试导出静态 PNG（如本机环境支持 kaleido）。

In [ ]:
name_fix = {
    "Czech Republic": "Czechia",
    "Macedonia (FYROM)": "North Macedonia",
    "Ivory Coast": "Cote d'Ivoire",
    "Vietnam": "Viet Nam",
    "Laos": "Lao People's Democratic Republic",
    "Tanzania": "Tanzania",
}

map_df = country_by_revenue.copy()
map_df = map_df[map_df["country"].notna()].copy()
map_df["country_plot"] = map_df["country"].map(lambda x: name_fix.get(x, x))

fig = px.choropleth(
    map_df,
    locations="country_plot",
    locationmode="country names",
    color="revenue_usd",
    hover_name="country",
    hover_data={"revenue_usd": ":.2f", "transactions": True, "sessions": True},
    color_continuous_scale="Blues",
    title="Transaction Revenue by Country (USD)",
)
fig.update_layout(margin=dict(l=10, r=10, t=50, b=10))

html_path = unique_path(FIG_DIR, "country_revenue_world_map", ".html")
fig.write_html(html_path, include_plotlyjs="cdn")
print("Saved HTML map ->", html_path)

png_path = unique_path(FIG_DIR, "country_revenue_world_map", ".png")
try:
    fig.write_image(png_path, width=1200, height=650, scale=2)
    print("Saved PNG map ->", png_path)
except Exception as exc:
    fail_path = unique_path(LOG_DIR, "world_map_png_export_failed", ".txt")
    fail_path.write_text(str(exc), encoding="utf-8")
    print("PNG export failed; wrote error ->", fail_path)


## 9. 导出产物与运行日志

导出内容：
- CSV 汇总表（国家/大洲、映射表）到 `tables/`
- PNG/HTML 图表到 `figures/`
- 运行元信息、字段选择、导出失败原因等到 `logs/`

说明：Notebook 运行时生成的文件名均带时间戳，避免覆盖历史产物。

In [ ]:
total_revenue = float(_df["revenue_usd"].fillna(0).sum())
total_txn = float(_df["transactions"].fillna(0).sum())
paid_sessions = int((_df["revenue_usd"].fillna(0) > 0).sum())

# Top3 countries for quick report copy
_top3 = country_by_revenue.head(3)[["country", "revenue_usd", "transactions", "sessions"]].to_dict(orient="records")

run_meta = {
    "run_timestamp": run_ts,
    "repo_root": str(REPO_ROOT),
    "cleaned_dir": str(CLEANED_DIR),
    "date_range": {
        "min": None if pd.isna(_df["date"].min()) else str(_df["date"].min().date()),
        "max": None if pd.isna(_df["date"].max()) else str(_df["date"].max().date()),
    },
    "sessions": int(len(_df)),
    "paid_sessions": paid_sessions,
    "total_revenue_usd": total_revenue,
    "total_transactions": total_txn,
    "top3_countries": _top3,
}

meta_path = write_json(run_meta, LOG_DIR, "run_metadata")
print("Saved run metadata ->", meta_path)

print("\nQuick summary:")
print("- date range:", run_meta["date_range"]["min"], "~", run_meta["date_range"]["max"])
print("- total revenue (USD):", round(total_revenue, 2))
print("- paid sessions:", paid_sessions)
print("- top3:")
for r in _top3:
    print("  ", r["country"], "revenue=", round(r["revenue_usd"], 2), "txns=", r["transactions"], "sessions=", r["sessions"])
